In [11]:
import pandas as pd
import helpy
import numpy as np
import importlib
from scipy.stats import ttest_ind

In [12]:
importlib.reload(helpy)

<module 'helpy' from '/home/lemonan/lppm_plus/helpy.py'>

In [13]:
analysis_df = helpy.get_analysis_df(.15, 3.8, 2, 'SIERA')

In [14]:
analysis_df['SIERA'].median()

np.float64(4.049442152836433)

In [15]:
pitch_value_cols = [
    "piwFA/C", "piwFC/C", "piwFS/C",
    "piwSI/C", "piwCH/C", "piwSL/C", "piwCU/C"
]

usage_cols = [
    "piFA%", "piFC%", "piFS%",
    "piSI%", "piCH%", "piSL%", "piCU%"
]

# Replace with your actual total pitches column
total_pitches_col = 'Pitches'

pitch_counts = pd.DataFrame(index=analysis_df.index)

for usage_col, value_col in zip(usage_cols, pitch_value_cols):
    pitch_counts[value_col] = (
        analysis_df[total_pitches_col] * analysis_df[usage_col]
    )

min_pitch_count = 50

pitch_values = analysis_df[pitch_value_cols].copy()

for col in pitch_value_cols:
    pitch_values.loc[
        pitch_counts[col] < min_pitch_count,
        col
    ] = np.nan

# Calculate metrics using only qualifying pitches
analysis_df['best_pitch_value'] = pitch_values.max(axis=1)

analysis_df['worst_pitch_value'] = pitch_values.min(axis=1)

analysis_df['second_best_pitch_value'] = pitch_values.apply(
    lambda row: row.dropna().sort_values(ascending=False).iloc[1]
    if row.notna().sum() >= 2 else np.nan,
    axis=1
)

analysis_df['pitch_rv_std'] = pitch_values.std(axis=1)

analysis_df['pitch_rv_range'] = (
    analysis_df['best_pitch_value'] -
    analysis_df['worst_pitch_value']
)

In [16]:
led_to_breakout_seasons = analysis_df.loc[analysis_df['led_to_breakout']].copy()
not_led_to_breakout_seasons = analysis_df.loc[~analysis_df['led_to_breakout']].copy()

In [17]:
led_to_breakout_seasons[['pitch_rv_std', 'pitch_rv_range', 'best_pitch_value', 'second_best_pitch_value', 'worst_pitch_value']].describe()

,pitch_rv_std,pitch_rv_range,best_pitch_value,second_best_pitch_value,worst_pitch_value
count,218.000000,220.000000,220.000000,218.000000,220.000000
mean,0.981460,2.268867,1.009390,0.361562,-1.259477
std,0.852675,1.988738,0.886359,0.637340,1.541401
min,0.058357,0.000000,-0.376972,-1.395668,-10.075059
25%,0.425472,0.975762,0.393225,0.022287,-1.575075
50%,0.705244,1.670891,0.644483,0.271652,-0.774816
75%,1.307262,3.114958,1.478059,0.599422,-0.380388
max,7.013549,13.972407,4.909424,3.048642,1.415088


In [18]:
not_led_to_breakout_seasons[['pitch_rv_std', 'pitch_rv_range', 'best_pitch_value', 'second_best_pitch_value', 'worst_pitch_value']].describe()

,pitch_rv_std,pitch_rv_range,best_pitch_value,second_best_pitch_value,worst_pitch_value
count,1647.000000,1650.000000,1650.000000,1647.000000,1650.000000
mean,0.938304,2.229759,1.052329,0.446490,-1.177430
std,0.676394,1.651893,0.879967,0.667977,1.321700
min,0.004955,0.000000,-5.451960,-3.582348,-9.929008
25%,0.447054,1.050885,0.451630,0.093439,-1.631192
50%,0.728362,1.697003,0.792569,0.346363,-0.810089
75%,1.278799,3.019002,1.509538,0.740148,-0.357566
max,5.080766,11.125516,5.952867,4.195307,1.717094


In [19]:
df = analysis_df[['PlayerName', 'led_to_breakout', 'pitch_rv_std']].dropna().copy()

# Calculate observed difference
breakout_mean = df.loc[df['led_to_breakout'], 'pitch_rv_std'].mean()
non_breakout_mean = df.loc[~df['led_to_breakout'], 'pitch_rv_std'].mean()

observed_diff = breakout_mean - non_breakout_mean

n_bootstrap = 5000
rng = np.random.default_rng(42)

# Convert to arrays
values = df['pitch_rv_std'].to_numpy()
groups = df['PlayerName'].to_numpy()
labels = df['led_to_breakout'].to_numpy()

pitchers = np.unique(groups)

# Map each row to a pitcher index
group_codes = pd.Categorical(groups, categories=pitchers).codes

bootstrap_diffs = []

for _ in range(n_bootstrap):

    sampled = rng.choice(pitchers, size=len(pitchers), replace=True)

    # Count how many times each pitcher was sampled
    counts = np.bincount(
        np.searchsorted(pitchers, sampled),
        minlength=len(pitchers)
    )

    weights = counts[group_codes]

    breakout_mask = labels & (weights > 0)
    non_breakout_mask = ~labels & (weights > 0)

    breakout_weights = weights[labels]
    non_breakout_weights = weights[~labels]

    b_mask = labels
    nb_mask = ~labels

    b_sum = np.sum(values[b_mask] * weights[b_mask])
    b_count = np.sum(weights[b_mask])

    nb_sum = np.sum(values[nb_mask] * weights[nb_mask])
    nb_count = np.sum(weights[nb_mask])

    if b_count > 0 and nb_count > 0:
        bootstrap_diffs.append(
            b_sum / b_count - nb_sum / nb_count
        )

ci_lower, ci_upper = np.percentile(bootstrap_diffs, [2.5, 97.5])

print(f"Observed difference: {observed_diff:.3f}")
print(f"95% CI: ({ci_lower:.3f}, {ci_upper:.3f})")

Observed difference: 0.043
95% CI: (-0.079, 0.177)
